# Assignment 07 — ST: forecasting stock volatility with a 5-layer RNN and LSTM

**Dataset:** *S&P 500 Stocks* (andrewmvd, Kaggle): daily open, high, low, close, adjusted close and volume of every S&P 500 member, 2010-01-04 → 2024-12-20 (96 MB CSV, 1.89M rows).
**Task:** from the last $T=60$ trading days of a stock, predict the size of its **next-day move**, the absolute log return $|r_{t+1}|$ (a daily volatility forecast).

**Why volatility and not the price direction?** Daily returns are close to a random walk: yesterday's return says almost nothing about tomorrow's sign (Assignment 06 measured exactly that). The *size* of moves is different: calm days follow calm days and turbulent days follow turbulent days (**volatility clustering**, shown in §3). So there is real temporal structure to learn, and a fair test of whether a model with longer memory (LSTM) uses it better than a plain RNN.

**Models:** {5-layer RNN, 5-layer LSTM} × {Keras, PyTorch}, 32 units per layer, linear output. Run IDs `ST-TF-RNN`, `ST-TF-LSTM`, `ST-PT-RNN`, `ST-PT-LSTM`. Structures: `00_rnn_lstm_concepts.ipynb`.

## 1. Setup

Same rules for every run: the same window arrays, architecture, Adam (lr $10^{-3}$, clip-norm 1), batch 256, at most 10 epochs, early stopping with patience 2 on validation loss (best epoch kept), seed 42.

In [ ]:
import os, time, json, random, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA = Path("../dataset/st")
RES = Path("results"); RES.mkdir(parents=True, exist_ok=True)
IMG = Path("../report/images"); IMG.mkdir(parents=True, exist_ok=True)

T, H, LAYERS = 60, 32, 5
BATCH, LR, CLIP = 256, 1e-3, 1.0
EPOCHS, PATIENCE = 10, 2
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
print("results ->", RES.resolve())

### Hyperparameters

| Name | Value | Meaning |
|---|---|---|
| stocks | all with the full 2010–2024 history (150) | stocks that joined or left the index later are left out, so every stock covers the same dates |
| window $T$ | 60 trading days | about three months of history per prediction |
| features $F$ | 4 per day | log return, high–low range, open–close return, log volume (z-scored) |
| target | $\lvert r_{t+1}\rvert$ | absolute log return of the next day |
| hidden units $H$ | 32 per layer, 5 layers | |
| loss | mean squared error on the standardised target | |
| optimizer | Adam, lr $10^{-3}$, clip-norm 1 | |
| split | chronological 70 / 15 / 15 by date | the same cut dates for every stock |

## 2. Data

The file is in *long* format: one row per (date, stock). Rows before a company joined the index are empty, so only stocks with a **complete** history are kept (all of them, ordered by mean dollar volume).

In [ ]:
raw = pd.read_csv(DATA / "sp500_stocks.csv", parse_dates=["Date"])
print("raw rows:", f"{len(raw):,}", "| stocks:", raw.Symbol.nunique(), "| dates:", raw.Date.min().date(), "->", raw.Date.max().date())
display(raw.dropna().head(3))
print("empty price rows:", f"{raw.Close.isna().sum():,}")

In [ ]:
px = raw.dropna(subset=["Close", "Open", "High", "Low", "Volume"])
px = px[(px.Close > 0) & (px.Volume > 0)]
n_days = px.Date.nunique()
full = px.groupby("Symbol").Date.count()
full = full[full == n_days].index
dollar_vol = (px[px.Symbol.isin(full)].assign(dv=lambda d: d.Close * d.Volume).groupby("Symbol").dv.mean().sort_values(ascending=False))
TICKERS = list(dollar_vol.index)                                   # all stocks with the full history, most traded first
px = px[px.Symbol.isin(TICKERS)].sort_values(["Symbol", "Date"])
print(f"{len(full)} stocks have all {n_days} trading days; all {len(TICKERS)} are used:")
print(", ".join(TICKERS))

### Features, step by step

For stock $s$ on day $t$ (adjusted close $P_t$ accounts for splits and dividends):

| Feature | Formula | Example |
|---|---|---|
| log return | $r_t=\ln(P_t/P_{t-1})$ | $P$: 100 → 102 gives $\ln1.02=0.0198$ (≈ +2 %) |
| high–low range | $(\text{High}-\text{Low})/\text{Close}$ | high 103, low 99, close 102 → 0.039 |
| open–close return | $\ln(\text{Close}/\text{Open})$ | open 101, close 102 → 0.0098 |
| log volume | $\ln(\text{Volume})$, z-scored per stock with train statistics | |
| **target** | $y_t=\lvert r_{t+1}\rvert$ | tomorrow 102 → 99.5: $\lvert\ln(99.5/102)\rvert=0.0248$ |

The log return is used because log returns add up over days and are symmetric (+ and − moves of the same size have the same magnitude).

In [ ]:
px["ret"] = px.groupby("Symbol")["Adj Close"].transform(lambda s: np.log(s).diff())
px["hl"] = (px.High - px.Low) / px.Close
px["oc"] = np.log(px.Close / px.Open)
px["lvol"] = np.log(px.Volume)
px["target"] = px.groupby("Symbol").ret.shift(-1).abs()          # |r_{t+1}|: known only tomorrow
px = px.dropna(subset=["ret", "target"])
dates = np.sort(px.Date.unique())
c1, c2 = dates[int(0.70 * len(dates))], dates[int(0.85 * len(dates))]
print("train  :", pd.Timestamp(dates[0]).date(), "->", pd.Timestamp(c1).date())
print("valid  :", pd.Timestamp(c1).date(), "->", pd.Timestamp(c2).date())
print("test   :", pd.Timestamp(c2).date(), "->", pd.Timestamp(dates[-1]).date())
aapl = px[px.Symbol == TICKERS[0]].iloc[1:4]
display(aapl[["Date", "Symbol", "Adj Close", "ret", "hl", "oc", "lvol", "target"]])

## 3. Exploratory data analysis

Three plots: (1) prices of a few stocks with the split dates; (2) the distribution of daily returns, which has **fat tails** (big moves are far more common than a normal distribution predicts); (3) the key plot for this task: the autocorrelation of $r_t$ and of $\lvert r_t\rvert$.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
for s in TICKERS[:5]:
    d = px[px.Symbol == s]; ax[0].plot(d.Date, d["Adj Close"] / d["Adj Close"].iloc[0], lw=0.8, label=s)
for c in (c1, c2): ax[0].axvline(c, c="k", ls="--", lw=0.8)
ax[0].set_yscale("log"); ax[0].set_title("growth of $1 (log scale); dashed = train | valid | test"); ax[0].legend(fontsize=8)
r = px.ret.values
ax[1].hist(r, bins=200, range=(-0.1, 0.1), density=True, color="#2a78d6", label="daily log returns")
xx = np.linspace(-0.1, 0.1, 400); ax[1].plot(xx, np.exp(-xx**2 / (2 * r.var())) / np.sqrt(2 * np.pi * r.var()), "r", label="normal, same variance")
ax[1].set_yscale("log"); ax[1].legend(); ax[1].set_title(f"returns: std {r.std():.4f}, kurtosis {pd.Series(r).kurt():.1f} (normal = 0)")
plt.tight_layout(); plt.savefig(IMG / "ST_eda.png", bbox_inches="tight"); plt.show()

### Volatility clustering: why $\lvert r\rvert$ is predictable and $r$ is not

The autocorrelation at lag $k$, $\rho_k=\mathrm{corr}(z_t,z_{t-k})$, measures how much today resembles $k$ days ago. For returns $r_t$ it is ≈ 0 at every lag (no memory of direction). For absolute returns $\lvert r_t\rvert$ it is clearly positive and decays **slowly over many weeks**: long memory, the kind of dependence an LSTM is designed to keep.

In [ ]:
def acf(x, lags):
    x = x - x.mean(); v = (x * x).mean()
    return np.array([(x[k:] * x[:-k]).mean() / v for k in lags])
lags = np.arange(1, 61)
a_r = np.mean([acf(px[px.Symbol == s].ret.values, lags) for s in TICKERS], 0)
a_abs = np.mean([acf(np.abs(px[px.Symbol == s].ret.values), lags) for s in TICKERS], 0)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
ax[0].bar(lags, a_r, color="#999", label="returns r"); ax[0].bar(lags, a_abs, color="#d64545", alpha=0.7, label="absolute returns |r|")
ax[0].axhline(0, c="k", lw=0.5); ax[0].set_xlabel("lag k (days)"); ax[0].set_title("autocorrelation, averaged over stocks"); ax[0].legend()
d = px[px.Symbol == TICKERS[0]]
ax[1].plot(d.Date, d.ret, lw=0.4, c="#2a78d6"); ax[1].set_title(f"{TICKERS[0]}: daily returns, calm and turbulent periods cluster")
plt.tight_layout(); plt.savefig(IMG / "ST_clustering.png", bbox_inches="tight"); plt.show()
print("lag-1 autocorrelation: r = %.3f   |r| = %.3f   |r| at lag 60 = %.3f" % (a_r[0], a_abs[0], a_abs[-1]))

## 4. Windows, scaling and baselines

**Scaling.** The features and the target are z-scored, $z=(x-\mu)/\sigma$, with $\mu,\sigma$ computed on the **train period only** (log volume per stock, the others over all stocks). Using test-period statistics would leak future information.

**Windows.** For each stock the series of $F=4$ features becomes overlapping windows: days $t-59..t$ are the input, $\lvert r_{t+1}\rvert$ is the target. A window belongs to train, validation or test by the date of its **target**, so no target crosses a split boundary (a window may look back into the previous period, as it would in real use).
Example with $T=3$: features of days 1, 2, 3 → target $\lvert r_4\rvert$; days 2, 3, 4 → $\lvert r_5\rvert$; and so on.

In [ ]:
FEATS = ["ret", "hl", "oc", "lvol"]
tr_mask = px.Date < c1
px["lvol"] = (px.lvol - px[tr_mask].groupby("Symbol").lvol.transform("mean").groupby(px.Symbol).transform("first")) / \
             px[tr_mask].groupby("Symbol").lvol.transform("std").groupby(px.Symbol).transform("first")
mu = px.loc[tr_mask, ["ret", "hl", "oc"]].mean(); sd = px.loc[tr_mask, ["ret", "hl", "oc"]].std()
for f in ["ret", "hl", "oc"]: px[f] = (px[f] - mu[f]) / sd[f]
y_mu, y_sd = px.loc[tr_mask, "target"].mean(), px.loc[tr_mask, "target"].std()

def make_windows(d):
    F = d[FEATS].values.astype("float32"); y = d.target.values.astype("float32"); dt = d.Date.values
    raw_abs = np.abs(d.ret.values * sd["ret"] + mu["ret"])                     # |r_t| in original units, for baselines
    idx = np.arange(T - 1, len(d))
    X = np.stack([F[i - T + 1:i + 1] for i in idx])
    return X, y[idx], dt[idx], raw_abs[idx], np.array([raw_abs[max(0, i - 19):i + 1].mean() for i in idx])

parts = {k: [] for k in ["X", "y", "date", "abs_now", "abs_mean20", "sym"]}
for s in TICKERS:
    X, y, dt, a_now, a_m20 = make_windows(px[px.Symbol == s])
    for k, v in zip(["X", "y", "date", "abs_now", "abs_mean20"], [X, y, dt, a_now, a_m20]): parts[k].append(v)
    parts["sym"].append(np.full(len(y), s))
A = {k: np.concatenate(v) for k, v in parts.items()}
split = np.where(A["date"] < c1, 0, np.where(A["date"] < c2, 1, 2))
ys = (A["y"] - y_mu) / y_sd                                                        # standardised target
Xtr, Xva, Xte = (A["X"][split == i] for i in range(3))
ytr, yva, yte_s = (ys[split == i] for i in range(3))
yte = A["y"][split == 2]                                                           # test target in original units
print(f"windows: train {len(Xtr):,}  valid {len(Xva):,}  test {len(Xte):,}   X shape {Xtr.shape} = (windows, T, F)")

### Metrics and baselines

Errors are reported in the original unit (absolute daily log return, shown in %): RMSE $=\sqrt{\frac1N\sum(y-\hat y)^2}$, MAE $=\frac1N\sum\lvert y-\hat y\rvert$, the out-of-sample $R^2=1-\frac{\sum(y-\hat y)^2}{\sum(y-\bar y_{\text{train}})^2}$ (0 = no better than always predicting the train mean) and the correlation between forecast and truth.

Baselines a model must beat:
- **train mean**: always predict the average $\lvert r\rvert$ of the train period;
- **persistence**: tomorrow's move = today's move, $\hat y=\lvert r_t\rvert$;
- **20-day mean**: the average $\lvert r\rvert$ of the last 20 days (a classical volatility estimate).

In [ ]:
RUNS = {}
def metrics(y, p):
    e = y - p
    return dict(rmse=float(np.sqrt((e ** 2).mean())), mae=float(np.abs(e).mean()),
                r2_oos=float(1 - (e ** 2).sum() / ((y - y_mu) ** 2).sum()), corr=float(np.corrcoef(y, p)[0, 1]))
BASE = {"train mean": metrics(yte, np.full_like(yte, y_mu)),
        "persistence |r_t|": metrics(yte, A["abs_now"][split == 2]),
        "20-day mean |r|": metrics(yte, A["abs_mean20"][split == 2])}
display((pd.DataFrame(BASE).T * [100, 100, 1, 1]).rename(columns={"rmse": "RMSE (%)", "mae": "MAE (%)"}).round(4))

def finalize(run_id, p_std, history, params, sec_epoch, total_s):
    p = p_std * y_sd + y_mu                                                  # back to original units
    rec = dict(run_id=run_id, params=int(params), sec_per_epoch=round(sec_epoch, 2), train_seconds=round(total_s, 1),
               epochs=len(history["loss"]), best_epoch=int(np.argmin(history["val_loss"]) + 1), history=history, **metrics(yte, p))
    RUNS[run_id] = {**rec, "pred": p}
    json.dump(rec, open(RES / f"{run_id}.json", "w"), indent=1)
    print(f"{run_id}: RMSE {100 * rec['rmse']:.4f}%  MAE {100 * rec['mae']:.4f}%  R2 {rec['r2_oos']:.4f}  corr {rec['corr']:.3f}  "
          f"epochs {rec['epochs']} (best {rec['best_epoch']})  {sec_epoch:.0f} s/epoch")

def rnn_params(F, H, L, nb=1):  return F * H + H * H + nb * H + (L - 1) * (2 * H * H + nb * H)
EXPECTED = {"RNN": rnn_params(4, H, LAYERS) + H + 1, "LSTM": 4 * rnn_params(4, H, LAYERS) + H + 1}
print("expected parameters (Keras-style):", EXPECTED)

**Reading the baselines.** Persistence is *worse* than the train mean: a single day's move is a noisy estimate of tomorrow's. Averaging 20 days removes much of that noise and beats the mean: this is the volatility clustering of §3 put to work. The recurrent networks have to beat the 20-day mean to show that they learned something beyond a moving average.

## 5. Keras — runs `ST-TF-RNN`, `ST-TF-LSTM`

`Input((60, 4))` → 4 recurrent layers returning sequences → 1 returning its last state → `Dense(1)` with no activation (a regression output). Parameter count, RNN: layer 1 $4\cdot32+32\cdot32+32=1{,}184$, layers 2–5 $2\cdot32^2+32=2{,}080$ each, head 33, total $9{,}537$; LSTM: 4× the recurrent part, $38{,}049$.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

def build_keras(kind):
    cell = {"RNN": layers.SimpleRNN, "LSTM": layers.LSTM}[kind]
    m = keras.Sequential([layers.Input((T, len(FEATS)))] + [cell(H, return_sequences=(l < LAYERS - 1)) for l in range(LAYERS)] + [layers.Dense(1)],
                         name=f"keras_{kind}")
    m.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=CLIP), loss="mse")
    return m

class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self.t = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self.t)

def run_keras(kind):
    tf.keras.utils.set_random_seed(SEED)
    m = build_keras(kind); assert m.count_params() == EXPECTED[kind], (m.count_params(), EXPECTED[kind])
    timer = EpochTimer(); t0 = time.time()
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH, verbose=2,
              callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True), timer])
    p = m.predict(Xte, batch_size=2048, verbose=0)[:, 0]
    finalize(f"ST-TF-{kind}", p, {"loss": h.history["loss"], "val_loss": h.history["val_loss"]}, m.count_params(),
             float(np.median(timer.times[1:] or timer.times)), time.time() - t0)

build_keras("LSTM").summary()

In [ ]:
run_keras("RNN")

In [ ]:
run_keras("LSTM")

## 6. PyTorch — runs `ST-PT-RNN`, `ST-PT-LSTM`

`nn.RNN(4, 32, num_layers=5, batch_first=True)` / `nn.LSTM(...)` → last step of the top layer → `nn.Linear(32, 1)`, with Keras's initialisation (Glorot input weights, orthogonal recurrent weights, zero biases, LSTM forget bias 1) so both frameworks start alike. PyTorch's second bias vector adds $5\cdot32=160$ (RNN) or $5\cdot128=640$ (LSTM) parameters, all starting at zero.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(SEED)
DEV = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEV)

class TorchVol(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM}[kind](len(FEATS), H, num_layers=LAYERS, batch_first=True)
        self.head = nn.Linear(H, 1)
        for name, p in self.rnn.named_parameters():
            if name.startswith("weight_ih"):
                for blk in p.data.chunk(4 if kind == "LSTM" else 1): nn.init.xavier_uniform_(blk)
            elif name.startswith("weight_hh"):
                for blk in p.data.chunk(4 if kind == "LSTM" else 1): nn.init.orthogonal_(blk)
            else:
                nn.init.zeros_(p)
                if kind == "LSTM" and name.startswith("bias_ih"): p.data[H:2 * H] = 1.0
        nn.init.xavier_uniform_(self.head.weight); nn.init.zeros_(self.head.bias)
    def forward(self, x):                    # x: (B, 60, 4)
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(1)

def predict_torch(m, X, bs=4096):
    m.eval()
    with torch.no_grad():
        return np.concatenate([m(torch.from_numpy(X[i:i + bs]).to(DEV)).cpu().numpy() for i in range(0, len(X), bs)])

def run_torch(kind):
    torch.manual_seed(SEED)
    m = TorchVol(kind).to(DEV); n = sum(p.numel() for p in m.parameters())
    assert n == EXPECTED[kind] + LAYERS * H * (4 if kind == "LSTM" else 1)
    opt = torch.optim.Adam(m.parameters(), lr=LR, eps=1e-7); lossf = nn.MSELoss()
    Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr)
    hist = {"loss": [], "val_loss": []}; times = []; best, best_state, wait = np.inf, None, 0
    g = torch.Generator().manual_seed(SEED); t0 = time.time()
    for ep in range(EPOCHS):
        m.train(); te = time.time(); tot = 0.0
        for b in torch.randperm(len(Xt), generator=g).split(BATCH):
            xb, yb = Xt[b].to(DEV), yt[b].to(DEV)
            loss = lossf(m(xb), yb)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), CLIP); opt.step()
            tot += loss.item() * len(b)
        vl = float(((predict_torch(m, Xva) - yva) ** 2).mean()); times.append(time.time() - te)
        hist["loss"].append(tot / len(Xt)); hist["val_loss"].append(vl)
        print(f"epoch {ep + 1}: loss {hist['loss'][-1]:.4f}  val_loss {vl:.4f}  {times[-1]:.0f}s")
        if vl < best - 1e-4:
            best, wait = vl, 0; best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break
    m.load_state_dict(best_state)
    finalize(f"ST-PT-{kind}", predict_torch(m, Xte), hist, n, float(np.median(times[1:] or times)), time.time() - t0)

print(TorchVol("RNN"))

In [ ]:
run_torch("RNN")

In [ ]:
run_torch("LSTM")

## 7. Comparison

In [ ]:
rows = [{"run": k, **{m: v[m] for m in ["rmse", "mae", "r2_oos", "corr", "params", "sec_per_epoch", "epochs", "best_epoch"]}} for k, v in RUNS.items()]
rows += [{"run": f"baseline: {k}", **v} for k, v in BASE.items()]
table = pd.DataFrame(rows).set_index("run")
table[["rmse", "mae"]] *= 100
table = table.rename(columns={"rmse": "RMSE (%)", "mae": "MAE (%)"})
table.to_csv(RES / "ST_comparison.csv")
display(table.round(4))

### Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for k, v in RUNS.items():
    ax.plot(range(1, len(v["history"]["val_loss"]) + 1), v["history"]["val_loss"], "-" if "-TF-" in k else "--",
            c="C0" if k.endswith("RNN") else "C3", marker="o", label=k)
ax.set_xlabel("epoch"); ax.set_ylabel("validation MSE (standardised)"); ax.legend(); ax.set_title("validation loss")
plt.tight_layout(); plt.savefig(IMG / "ST_curves.png", bbox_inches="tight"); plt.show()

### Forecast against truth

The last 250 test days of the first stock: the true $\lvert r_{t+1}\rvert$ (grey), the 20-day-mean baseline and the Keras RNN and LSTM forecasts. A volatility forecast cannot predict *which* day will have a big move; a good one rises and falls with the turbulence level.

In [ ]:
s0 = TICKERS[0]; mask = A["sym"][split == 2] == s0
dt = A["date"][split == 2][mask][-250:]
plt.figure(figsize=(14, 3.8))
plt.plot(dt, 100 * yte[mask][-250:], c="#bbb", lw=0.8, label="true |r(t+1)|")
plt.plot(dt, 100 * A["abs_mean20"][split == 2][mask][-250:], c="k", lw=1, ls=":", label="20-day mean")
for k, c in [("ST-TF-RNN", "C0"), ("ST-TF-LSTM", "C3")]:
    if k in RUNS: plt.plot(dt, 100 * RUNS[k]["pred"][mask][-250:], c=c, lw=1.2, label=k)
plt.ylabel("% move"); plt.title(f"{s0}: next-day absolute return, test period"); plt.legend()
plt.tight_layout(); plt.savefig(IMG / "ST_forecast.png", bbox_inches="tight"); plt.show()

### Error in calm and turbulent periods

Test days split into five groups by the 20-day volatility level at prediction time. Memory matters most when the level changes; a model that only follows the last few days should lose more in the most turbulent group.

In [ ]:
lvl = pd.qcut(A["abs_mean20"][split == 2], 5, labels=["calmest", "calm", "middle", "turbulent", "most turbulent"])
err = pd.DataFrame({k: pd.Series((v["pred"] - yte) ** 2).groupby(lvl).mean() ** 0.5 * 100 for k, v in RUNS.items()})
err["20-day mean"] = pd.Series((A["abs_mean20"][split == 2] - yte) ** 2).groupby(lvl).mean() ** 0.5 * 100
display(err.round(4))
err.plot.bar(rot=0, figsize=(10, 3.6), title="test RMSE (%) by volatility regime"); plt.ylabel("RMSE (%)")
plt.tight_layout(); plt.savefig(IMG / "ST_by_regime.png", bbox_inches="tight"); plt.show()

## 8. Conclusion

Written after the full run (see the table in §7).